## Offside Archives

![log of app](logo.png)

### this app load all videos with their thumbnail even u can search on all videos

In [1]:
import tkinter as tk
from tkinter import messagebox
from PIL import Image, ImageTk
from pathlib import Path
import subprocess
import platform
import os

VIDEO_FOLDER = Path("MyVideos")

VIDEO_EXTENSIONS = [
    ".mp4",
    ".mkv",
    ".avi",
    ".mov",
    ".3gp"
]

class VideoBrowser:

    def __init__(self, root):
        self.root = root
        root.title("Offside Archives")  # تغییر عنوان برنامه به Offside
        root.geometry("1600x800")

        self.images = []

        # ایجاد Canvas و Scrollbar
        self.canvas = tk.Canvas(root)
        self.scrollbar = tk.Scrollbar(root, orient="vertical", command=self.canvas.yview)
        self.frame = tk.Frame(self.canvas)

        # تنظیم اسکرول بعد از تغییر اندازه فریم داخلی
        self.frame.bind(
            "<Configure>",
            lambda e: self.canvas.configure(scrollregion=self.canvas.bbox("all"))
        )

        self.canvas.create_window((0, 0), window=self.frame, anchor="nw")
        self.canvas.configure(yscrollcommand=self.scrollbar.set)

        self.canvas.pack(side="left", fill="both", expand=True)
        self.scrollbar.pack(side="right", fill="y")

        # فعال کردن اسکرول با چرخ موس روی کل پنجره
        self.canvas.bind_all("<MouseWheel>", self._on_mousewheel)

        # قرار دادن لوگو در بالاترین نقطه
        logo_path = Path(".\logo.png")
        if logo_path.exists():
            try:
                logo_img = Image.open(logo_path)
                logo_img.thumbnail((450, 400))
                self.logo_photo = ImageTk.PhotoImage(logo_img)
                
                self.logo_label = tk.Label(self.frame, image=self.logo_photo)
                self.logo_label.pack(pady=15)
            except Exception:
                self.logo_label = tk.Label(self.frame, text="Offside", font=("Arial", 24, "bold"))
                self.logo_label.pack(pady=15)
        else:
            self.logo_label = tk.Label(self.frame, text="Offside", font=("Arial", 24, "bold"))
            self.logo_label.pack(pady=15)

        # ---- بخش نوار جستجو با ذره‌بین و متن راهنما ----
        self.search_frame = tk.Frame(self.frame, padx=40, pady=10)
        self.search_frame.pack(fill="x")

        # نمایش آیکون ذره‌بین در سمت چپ کادر جستجو
        self.search_icon = tk.Label(self.search_frame, text="🔍", font=("Arial", 32))
        self.search_icon.pack(side="right", padx=(0, 10))

        # کادر ورود متن جستجو
        self.search_entry = tk.Entry(
            self.search_frame, 
            font=("Arial", 14), 
            bd=2, 
            relief="groove"
        )
        self.search_entry.pack(side="left", fill="x", expand=True, ipady=8)
        
        # ذخیره رنگ پیش‌فرض نوشته‌ها برای مدیریت متن راهنما
        self.default_fg = self.search_entry.cget("fg")
        
        # مقداردهی اولیه متن راهنما
        self.search_entry.insert(0, "Search...")
        self.search_entry.config(fg="grey")

        # اتصال رویدادهای کلیک و خارج شدن از کادر برای مدیریت متن راهنما
        self.search_entry.bind("<FocusIn>", self.on_focus_in)
        self.search_entry.bind("<FocusOut>", self.on_focus_out)
        self.search_entry.bind("<Return>", self.on_search)

        # ایجاد فریم داخلی برای چیدمان گرید ویدیوها
        self.grid_frame = tk.Frame(self.frame)
        self.grid_frame.pack(fill="both", expand=True)

        self.load_videos()

    def _on_mousewheel(self, event):
        if platform.system() == "Windows":
            self.canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")
        elif platform.system() == "Darwin": # macOS
            self.canvas.yview_scroll(int(-1 * event.delta), "units")
        else: # Linux
            if event.num == 4:
                self.canvas.yview_scroll(-1, "units")
            elif event.num == 5:
                self.canvas.yview_scroll(1, "units")

    def on_focus_in(self, event):
        # حذف متن راهنما هنگام کلیک روی کادر
        if self.search_entry.get() == "Search...":
            self.search_entry.delete(0, tk.END)
            self.search_entry.config(fg=self.default_fg)

    def on_focus_out(self, event):
        # بازگرداندن متن راهنما در صورت خالی بودن کادر
        if self.search_entry.get().strip() == "":
            self.search_entry.insert(0, "Search...")
            self.search_entry.config(fg="grey")

    def on_search(self, event=None):
        query = self.search_entry.get().strip()
        # اگر کاربر اینتر بزند و متن همچنان "جستجو..." باشد، آن را خالی در نظر می‌گیریم
        if query == "Search...":
            query = ""
        self.load_videos(search_term=query)

    def load_videos(self, search_term=""):
        # حذف کارت‌های قبلی
        for widget in self.grid_frame.winfo_children():
            widget.destroy()
        
        self.images = []

        if not VIDEO_FOLDER.exists():
            VIDEO_FOLDER.mkdir(parents=True, exist_ok=True)
            label = tk.Label(
                self.grid_frame, 
                text=f"پوشه '{VIDEO_FOLDER}' ایجاد شد.\nلطفاً ویدیوها و تصاویر PNG هم‌نام را در آن قرار دهید.", 
                padx=20, 
                pady=20,
                justify="center"
            )
            label.pack()
            return

        row = 0
        col = 0
        has_videos = False
        search_term_lower = search_term.lower()

        for video in VIDEO_FOLDER.iterdir():
            if video.suffix.lower() in VIDEO_EXTENSIONS:
                if search_term_lower and search_term_lower not in video.name.lower():
                    continue

                image_path = video.with_suffix(".png")

                if image_path.exists():
                    self.create_card(video, image_path, row, col)
                    has_videos = True
                    col += 1

                    if col == 3:
                        col = 0
                        row += 1

        if not has_videos:
            msg = f"ویدیویی با عبارت '{search_term}' یافت نشد." if search_term else "ویدیویی با تصویر پیش‌نمایش هم‌نام (با پسوند PNG) در پوشه یافت نشد."
            label = tk.Label(
                self.grid_frame, 
                text=msg, 
                padx=20, 
                pady=20
            )
            label.pack()

    def create_card(self, video, image_path, row, col):
        card = tk.Frame(self.grid_frame, padx=10, pady=10)

        try:
            img = Image.open(image_path)
            img.thumbnail((450, 300))
            photo = ImageTk.PhotoImage(img)
            self.images.append(photo)

            button = tk.Button(
                card,
                image=photo,
                command=lambda: self.open_video(video)
            )
            button.pack()
        except Exception:
            button = tk.Button(
                card,
                text="پیش‌نمایش در دسترس نیست",
                width=30,
                height=10,
                command=lambda: self.open_video(video)
            )
            button.pack()

        label = tk.Label(card, text=video.name, wraplength=450)
        label.pack()

        card.grid(row=row, column=col, padx=10, pady=10)

    def open_video(self, video):
        try:
            subprocess.Popen(["vlc", str(video)])
        except FileNotFoundError:
            try:
                if platform.system() == "Windows":
                    os.startfile(video)
                elif platform.system() == "Darwin":  # macOS
                    subprocess.Popen(["open", str(video)])
                else:  # Linux
                    subprocess.Popen(["xdg-open", str(video)])
            except Exception as e:
                messagebox.showerror("خطا", f"امکان پخش ویدیو وجود ندارد:\n{e}")

##### Run App with this cell

In [2]:
root = tk.Tk()
app = VideoBrowser(root)
root.mainloop()